# Продуктовая аналитика: Поиск точек роста бизнеса

**Ключевые показатели юнит-экономики:**
- **UA** (User Acquisition) — Общее количество входящих лидов.
- **C1** (Conversion Rate) — Конверсия из лида в уникального покупателя (B/UA).
- **B** (Buyers) — Количество уникальных подтверждённых покупателей.
- **T** (Transactions) — Общее количество подтверждённых сделок (оплат).
- **Rev** (Revenue) — Выручка (сумма начальных платежей подтверждённых покупателей).
- **APC** (Average Purchase Count) — Среднее количество покупок на одного покупателя (T/B).
- **AOV** (Average Order Value) — Средний чек за транзакцию (Rev/T).
- **AC** (Acquisition Cost) — Общие маркетинговые затраты.
- **CAC** (Customer Acquisition Cost) — Стоимость привлечения одного покупателя (AC/B).
- **CPA** (Cost Per Acquisition/Lead) — Стоимость одного лида (AC/UA). В коде часто `LTC`.
- **CLTV** (Customer Lifetime Value) — Доход от одного покупателя ($APC \times AOV$).
- **LTV** (Lifetime Value) — Доход от одного привлеченного лида ($C1 \times APC \times AOV = C1 \times CLTV$).
- **CM** (Contribution Margin) — Маржинальная прибыль (Rev - AC).

**Изменяемые рычаги для анализа чувствительности:** `UA`, `C1`, `APC`, `AOV`, `LTC`.


In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

import help_130625_dam  as h

pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

CLEANED_DIR = os.path.join('..', 'data', 'cleaned')

In [15]:
contacts      = pd.read_pickle(os.path.join(CLEANED_DIR, 'contacts_clean.pkl'))
deals         = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
calls         = pd.read_pickle(os.path.join(CLEANED_DIR, 'calls_clean.pkl'))
spend         = pd.read_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))
campaign_romi = pd.read_pickle(os.path.join(CLEANED_DIR, 'campaign_romi.pkl'))

print(f'Contacts:      {contacts.shape}')
print(f'Deals:         {deals.shape}')
print(f'Calls:         {calls.shape}')
print(f'Spend:         {spend.shape}')
print(f'Campaign ROMI: {campaign_romi.shape}')

Contacts:      (18510, 7)
Deals:         (19815, 28)
Calls:         (92599, 9)
Spend:         (19862, 8)
Campaign ROMI: (366, 12)


## Глобальная юнит-экономика

Сначала смотрим на бизнес целиком: как соотносятся затраты и доходы на уровне одного лида/клиента.

In [16]:
# Глобальные показатели юнит-экономики 

# UA = Уникальные контакты 
n_leads = int(contacts.shape[0])

# B = Уникальные покупатели (Люди, которые заплатили и учились)
# Условие: initial_amount_paid > 0 AND months_of_study > 0 AND stage == 'Payment Done'
buyer_mask = (deals['initial_amount_paid'] > 0) & (deals['months_of_study'] > 0) & (deals['stage'] == 'Payment Done')
n_buyers = int(deals[buyer_mask]['contact_id'].nunique())
print(f'   B (Уникальные покупатели) = {n_buyers}')
# T = Общее число подтверждённых транзакций 
buyer_deals = deals[buyer_mask]
n_transactions = len(buyer_deals)

# Rev = Выручка (Только по подтвержденным сделкам)
total_revenue = buyer_deals['initial_amount_paid'].sum()
total_spend = spend['spend'].sum()

# Основные юнит-метрики
crv_rate = n_buyers / n_leads if n_leads > 0 else 0
apc = n_transactions / n_buyers if n_buyers > 0 else 0
aov = total_revenue / n_transactions if n_transactions > 0 else 0
cac = total_spend / n_buyers if n_buyers > 0 else 0
ltc = total_spend / n_leads if n_leads > 0 else 0
cltv = apc * aov         
ltv = crv_rate * cltv     
profit = total_revenue - total_spend

global_ue = pd.DataFrame([{
    'UA (Contacts)': n_leads,
    'C1, %':         crv_rate,
    'B (Unique)':    n_buyers,
    'T (Deals)':     n_transactions,
    'APC':           apc,
    'Rev, €':        total_revenue,
    'AOV, €':        aov,
    'AC, €':         total_spend,
    'CAC, €':        cac,
    'CPA, €':        ltc,
    'CLTV, €':       cltv,
    'LTV, €':        ltv,
    'CM, €':         profit,
}])

display(
    global_ue.style
    .hide(axis='index')
    .format({
        'UA (Contacts)': '{:,.0f}',
        'C1, %':         '{:.2%}',
        'B (Unique)':    '{:,.0f}',
        'T (Deals)':     '{:,.0f}',
        'APC':           '{:.2f}',
        'Rev, €':        '{:,.0f}',
        'AOV, €':        '{:,.0f}',
        'AC, €':         '{:,.0f}',
        'CAC, €':        '{:,.0f}',
        'CPA, €':        '{:,.2f}',
        'CLTV, €':       '{:,.2f}',
        'LTV, €':        '{:,.2f}',
        'CM, €':         '{:,.0f}',
    })
    .map(
        lambda v: 'color: red; font-weight: bold' if isinstance(v, (int, float)) and v < 0 else '',
        subset=['CM, €']
    )
    .set_caption('Глобальная юнит-экономика (База: Люди)')
)

   B (Уникальные покупатели) = 814


UA (Contacts),"C1, %",B (Unique),T (Deals),APC,"Rev, €","AOV, €","AC, €","CAC, €","CPA, €","CLTV, €","LTV, €","CM, €"
"18,510",4.40%,814,826,1.01,"941,850","1,140","149,523",184,8.08,"1,157.06",50.88,"792,327"


#### Число транзакций на покупателя = 1.01 объясняется тем что после первой оплаты и начала обучения клиент становился неинтересен менеджеру и сделка закрывалась.

## Юнит-экономика по источникам трафика

In [17]:
# Юнит-экономика по продуктам (используем общие UA и AC)
# В анализе остаются только продукты с подтвержденными продажами (buyers > 0)

def product_agg(x):
    # Условие Buyer: Платеж > 0 + Обучение > 0 + Стадия Payment Done
    is_buyer_mask = (x['initial_amount_paid'] > 0) & (x['months_of_study'] > 0) & (x['stage'] == 'Payment Done')
    
    b = x[is_buyer_mask]['contact_id'].nunique()
    t = is_buyer_mask.sum()
    rev = x.loc[is_buyer_mask, 'initial_amount_paid'].sum()
    
    return pd.Series({
        'B':   b,
        'T':   t,
        'Rev': rev,
        'UA':  n_leads,
        'AC':  total_spend
    })

product_ue = (
    deals[deals['product'] != 'Unknown']
    .groupby('product', observed=True)
    .apply(product_agg, include_groups=False)
    .reset_index()
)

# Оставляем продукты с продажами
product_ue = product_ue[product_ue['B'] > 0].copy()

# Расчёт всех метрик
product_ue['c1']    = product_ue['B'] / product_ue['UA']
product_ue['apc']   = product_ue['T'] / product_ue['B']
product_ue['aov']   = product_ue['Rev'] / product_ue['T']
product_ue['cltv']  = product_ue['apc'] * product_ue['aov']        # Доход на покупателя (CLTV)
product_ue['ltv']   = product_ue['c1'] * product_ue['cltv']       # Доход на лид (LTV)
product_ue['cac']   = product_ue['AC'] / product_ue['B']
product_ue['cpa']   = product_ue['AC'] / product_ue['UA']
product_ue['cm']    = product_ue['Rev'] - product_ue['AC']

product_ue = product_ue.sort_values('Rev', ascending=False).set_index('product')

print(f'Справочно: Продуктов с продажами: {len(product_ue)} | Глобальный UA = {n_leads:,.0f} | Глобальный AC = {total_spend:,.0f} €')

display(
    product_ue.style
    .format({
        'B': '{:,.0f}', 'T': '{:,.0f}', 'UA': '{:,.0f}',
        'Rev': '{:,.0f} €', 'AC': '{:,.0f} €', 'cm': '{:,.0f} €',
        'c1': '{:.2%}', 'apc': '{:.2f}', 'aov': '{:,.0f} €',
        'cltv': '{:,.2f} €', 'ltv': '{:,.2f} €', 'cac': '{:,.0f} €', 'cpa': '{:.2f} €'
    })
    .map(lambda v: 'color: red; font-weight: bold' if v < 0 else 'color: green', subset=['cm'])
    .set_caption('Юнит-экономика продуктов (UA = Контакты)')
)


Справочно: Продуктов с продажами: 3 | Глобальный UA = 18,510 | Глобальный AC = 149,523 €


,B,T,Rev,UA,AC,c1,apc,aov,cltv,ltv,cac,cpa,cm
product,,,,,,,,,,,,,
Digital Marketing,463,468,"539,100 €","18,510","149,523 €",2.50%,1.01,"1,152 €","1,164.36 €",29.12 €,323 €,8.08 €,"389,577 €"
UX/UI Design,218,221,"259,300 €","18,510","149,523 €",1.18%,1.01,"1,173 €","1,189.45 €",14.01 €,686 €,8.08 €,"109,777 €"
Web Developer,135,137,"143,450 €","18,510","149,523 €",0.73%,1.01,"1,047 €","1,062.59 €",7.75 €,"1,108 €",8.08 €,"-6,073 €"


In [18]:
# Анализ чувствительности ПО ПРОДУКТАМ (±10% для каждого)
# Изменяем рычаги: UA, C1, APC, AOV, LTC (CPA)

delta = 0.10
sens_results = []

# Берем топ-5 продуктов по выручке для анализа
top_products = product_ue.head(5).index.tolist()

for prod in top_products:
    row = product_ue.loc[prod]
    # Базовые значения
    ua0   = row['UA']
    c10   = row['c1']
    apc0  = row['apc']
    aov0  = row['aov']
    ltc0  = row['cpa']
    
    # Сценарии (изменения только ОДНОЙ метрики за раз)
    scenarios = [
        ('1. Факт (база)',  ua0, c10, apc0, aov0, ltc0),
        ('2. +10% UA',      ua0 * (1 + delta), c10, apc0, aov0, ltc0),
        ('3. +10% C1',      ua0, c10 * (1 + delta), apc0, aov0, ltc0),
        ('4. +10% APC',     ua0, c10, apc0 * (1 + delta), aov0, ltc0),
        ('5. +10% AOV',     ua0, c10, apc0, aov0 * (1 + delta), ltc0),
        ('6. -10% LTC (CPA)', ua0, c10, apc0, aov0, ltc0 * (1 - delta)),
    ]
    
    for label, ua, c1, apc, aov, ltc in scenarios:
        # Расчет в глубину
        b_new    = ua * c1
        t_new    = b_new * apc
        rev_new  = t_new * aov
        ac_new   = ua * ltc
        cltv_new = apc * aov
        ltv_new  = c1 * cltv_new
        cm_new   = rev_new - ac_new
        
        sens_results.append({
            'Продукт':   prod,
            'Сценарий':  label,
            'UA':        ua,
            'C1':        c1,
            'APC':       apc,
            'AOV':       aov,
            'LTC':       ltc,
            'CLTV, €':   cltv_new,
            'LTV, €':    ltv_new,
            'Rev, €':    rev_new,
            'AC, €':     ac_new,
            'CM, €':     cm_new
        })

sens_prod_df = pd.DataFrame(sens_results)

# Считаем ΔCM относительно базы
final_list = []
for prod, group in sens_prod_df.groupby('Продукт', observed=True):
    base_cm = group.iloc[0]['CM, €']
    group['ΔCM, €'] = group['CM, €'] - base_cm
    group.iloc[0, group.columns.get_loc('ΔCM, €')] = np.nan
    final_list.append(group)

sens_prod_df = pd.concat(final_list)

display(
    sens_prod_df.set_index(['Продукт', 'Сценарий']).style
    .format({
        'UA':        '{:,.0f}',
        'C1':        '{:.2%}',
        'APC':       '{:.2f}',
        'AOV':       '{:,.0f}',
        'LTC':       '{:.2f}',
        'CLTV, €':   '{:,.2f}',
        'LTV, €':    '{:,.2f}',
        'Rev, €':    '{:,.0f}',
        'AC, €':     '{:,.0f}',
        'CM, €':     '{:,.0f}',
        'ΔCM, €':    '{:+,.0f}',
    }, na_rep='—')
    .map(
        lambda v: 'color: red' if isinstance(v, (int, float)) and v < 0 else '',
        subset=['CM, €']
    )
    .set_caption(f'Анализ чувствительности: Рычаги роста (±{delta:.0%})')
)


## Продуктовые гипотезы и A/B тесты

Для трех ключевых продуктов сформированы уникальные гипотезы, направленные на рост конверсии первого этапа (C1):

────────────────────────────────────────────────────────────────────────

**[H_DM] Digital Marketing: Калькулятор ROI обучения**
- **Гипотеза:** Добавление в оффер калькулятора (прогноз зарплаты через 6-12 месяцев против стоимости обучения) увеличит C1 на 15%.
- **Обоснование:** Снижение страха "потери денег" за счет демонстрации окупаемости инвестиций.

**[H_UX] UX/UI Design: Визуальный лид-магнит (Figma Workshop)**
- **Гипотеза:** Замена стандартного вводного звонка на приглашение на мини-воркшоп "Первый проект в Figma за 40 минут" увеличит C1 на 15%.
- **Обоснование:** Клиент сразу получает "быструю победу" и вовлекается в продукт через практику.

**[H_Web] Web Developer: Технический скрининг (Code Review)**
- **Гипотеза:** Предложение бесплатной проверки кода или технической консультации на первом этапе увеличит C1 на 15%.
- **Обоснование:** Формирование экспертного доверия. Клиент видит в школе не просто "продавцов", а профессиональную среду.

────────────────────────────────────────────────────────────────────────


In [26]:
# План A/B-тестирования: По одной уникальной гипотезе для каждого продукта

alpha = 0.05
power = 0.8
delta = 0.15 

product_hypotheses = [
    {
        "Продукт": "Digital Marketing",
        "Гипотеза": "[H_DM] Добавление в оффер калькулятора ROI обучения (зарплата через 6 мес) увеличит C1 на 15%",
        "Baseline (C1)": product_ue.loc["Digital Marketing", "c1"] if "Digital Marketing" in product_ue.index else crv_rate,
        "MDE": delta,
        "Тип": "ROI Calc"
    },
    {
        "Продукт": "UX/UI Design",
        "Гипотеза": "[H_UX] Замена вводного звонка на мини-воркшоп 'Первый проект в Figma' увеличит C1 на 15%",
        "Baseline (C1)": product_ue.loc["UX/UI Design", "c1"] if "UX/UI Design" in product_ue.index else crv_rate,
        "MDE": delta,
        "Тип": "Workshop"
    },
    {
        "Продукт": "Web Developer",
        "Гипотеза": "[H_Web] Предложение бесплатного технического аудита (code review) на первом этапе увеличит C1 на 15%",
        "Baseline (C1)": product_ue.loc["Web Developer", "c1"] if "Web Developer" in product_ue.index else crv_rate,
        "MDE": delta,
        "Тип": "Code Review"
    }
]

# Расчет объема выборки для каждой гипотезы
ab_product_results = []
# Среднее кол-во лидов в день на ВЕСЬ проект (т.к. лиды общие)
daily_leads_global = n_leads / ((deals['created_time'].max() - deals['created_time'].min()).days + 1)

for h_data in product_hypotheses:
    p1 = h_data["Baseline (C1)"]
    p2 = p1 * (1 + h_data["MDE"])
    
    n_per_group = h.ab_sample_size(p1, h_data["MDE"], alpha=alpha, power=power)
    n_total = n_per_group * 2
    days_needed = n_total / daily_leads_global
    
    ab_product_results.append({
        "Продукт": h_data["Продукт"],
        "Гипотеза": h_data["Гипотеза"],
        "Базовый C1": f"{p1:.2%}",
        "Целевой C1": f"{p2:.2%}",
        "Выборка (всего)": int(n_total),
        "Дней теста": f"{round(days_needed, 0):.0f}",
        "Реализуемо за 14д?": "Да" if days_needed <= 14 else "Нет"
    })

ab_comparison_df = pd.DataFrame(ab_product_results)

display(ab_comparison_df.style.set_caption("Сравнение длительности тестов по продуктовым гипотезам"))


,Продукт,Гипотеза,Базовый C1,Целевой C1,Выборка (всего),Дней теста,Реализуемо за 14д?
0,Digital Marketing,[H_DM] Добавление в оффер калькулятора ROI обучения (зарплата через 6 мес) увеличит C1 на 15%,2.50%,2.88%,58286,1115,Нет
1,UX/UI Design,[H_UX] Замена вводного звонка на мини-воркшоп 'Первый проект в Figma' увеличит C1 на 15%,1.18%,1.35%,125600,2402,Нет
2,Web Developer,[H_Web] Предложение бесплатного технического аудита (code review) на первом этапе увеличит C1 на 15%,0.73%,0.84%,203808,3898,Нет


### Проблема скорости тестов (Низкий трафик)

Если мы продолжим измерять «Деньги» (финальную оплату с C1 ≈ 4%), то при текущем трафике (~56 лидов/день) тест с MDE 10% будет идти 1371 день (3.5 года). Это слишком долго.

> **Решение:** Переход от финальных метрик (Деньги) к прокси-метрикам (Действия) с более высокой базовой конверсией.

### Новые гипотезы 

#### [H4] Автоматизация первого касания
- **Суть:** Внедрение WhatsApp-бота, который пишет лиду в течение 2 минут после регистрации.
- **Прокси-метрика:** Конверсия в «Успешный контакт / Дозвон» (ожидаем рост с 50% до 65%).
- **Зачем:** Нулевой SLA гарантированно повышает лояльность и вероятность продажи.

#### [H5] Лид-магнит и фокус на Интенсив 
- **Суть:** Вместо продажи курса «в лоб», предлагать бесплатный интенсив/пробные уроки.
- **Прокси-метрика:** Конверсия в «Запись на интенсив» (ожидаем базовую конверсию ~30%).
- **Зачем:** Снимаем барьер первого платежа. 

---
#### Расчет времени проверки гипотез:


In [27]:
# Сравнение времени теста для разных гипотез
p_h1 = 0.041
p_h4 = 0.50          # Прокси: успешный контакт
p_h5 = 0.30          # Прокси: запись на бесплатный продукт

mde_h1 = 0.10        # Ожидаем +10% роста оплат
mde_h4 = 0.40        # Ожидаем +40% роста дозвона (автоматизация) 
mde_h5 = 0.50        # Ожидаем +50% записи (бесплатный продукт вместо платного)

# Используем ранее рассчитанный daily_leads_global
plans = []
for label, p, mde in [
    ("H1: SLA (Оплата)", p_h1, mde_h1),
    ("H4: Auto-touch (Контакт)", p_h4, mde_h4), 
    ("H5: Lead-Magnet (Запись)", p_h5, mde_h5)
]:
    n_group = h.ab_sample_size(p, mde)
    n_total = n_group * 2
    # Используем GLOBAL traffic (все 56 лидов в день на тест)
    days = n_total / daily_leads_global
    
    plans.append({
        "Гипотеза": label,
        "Базовая конф.": f"{p:.1%}",
        "MDE (отн.)": f"+{mde:.0%}",
        "Цель": f"{p*(1+mde):.1%}",
        "Выборка (всего)": int(n_total),
        "Дней теста": round(days, 1),
        "Реализуемо за 14д?": " ДА" if days <= 14 else " НЕТ"
    })

comparison_df = pd.DataFrame(plans)

# Оставляем только красивый вывод через .style
display(comparison_df.style.set_caption("Сравнение длительности тестов: Деньги против действий"))

,Гипотеза,Базовая конф.,MDE (отн.),Цель,Выборка (всего),Дней теста,Реализуемо за 14д?
0,H1: SLA (Оплата),4.1%,+10%,4.5%,76900,1470.700000,НЕТ
1,H4: Auto-touch (Контакт),50.0%,+40%,70.0%,186,3.600000,ДА
2,H5: Lead-Magnet (Запись),30.0%,+50%,45.0%,324,6.200000,ДА


In [28]:
# Расчет воронки и конверсии по SLA для отчета
funnel = deals['stage'].value_counts().to_frame().rename(columns={'count': 'leads'})

# Конверсия в зависимости от SLA (генерация данных, если они не были созданы ранее)
if 'sla_group' in deals.columns:
    sla_cr = deals.groupby('sla_group', observed=True)['is_buyer'].mean().to_frame().rename(columns={'is_buyer': 'cr'})
else:
    # Заглушка, если колонка не найдена (предотвращает ошибку в финальной ячейке)
    sla_cr = pd.DataFrame({'cr': [crv_rate]}, index=['Global'])

if 'quality' in deals.columns:
    q_cr = deals.groupby('quality', observed=True)['is_buyer'].mean().to_frame().rename(columns={'is_buyer': 'cr'})
else:
    q_cr = pd.DataFrame()


In [29]:
# Сохранение данных для презентации (08_presentation.ipynb) 
# Все ключевые агрегаты пакуем в один словарь → report_data.pkl

REPORT_DATA_PATH = os.path.join(CLEANED_DIR, 'report_data.pkl')

# Новые гипотезы (Прокси-метрики), которые РЕАЛЬНО провести за 14 дней
hypotheses_v3 = comparison_df.to_dict('records')

report_data = {
    # Глобальные KPI
    'global_kpi': {
        'n_leads':       n_leads,
        'n_buyers':      n_buyers,
        'crv_rate':      crv_rate,
        'avg_ltv':       ltv,    
        'cac':           cac,
        'total_revenue': total_revenue,
        'total_spend':   total_spend,
        'profit':        profit,
        'roas':          total_revenue / total_spend,
        'apc':           apc,
        'cltv':          cltv,
        'ltv':           ltv,
        'cpa':           ltc
    },

    # Таблицы юнит-экономики
    'product_ue': product_ue.head(15).reset_index(),

    # Анализ чувствительности (Рычаги роста)
    'sens_df': sens_prod_df.reset_index(),

    # Воронка и гипотезы
    'funnel': funnel.reset_index(),
    'sla_cr': sla_cr.reset_index(),
    'q_cr':   q_cr.reset_index() if 'q_cr' in locals() else pd.DataFrame(),

    # Продуктовые гипотезы (передаем только названия для справки о невозможности теста)
    'product_hypotheses_names': [h['Продукт'] for h in product_hypotheses],
    
    # Рекомендуемые гипотезы (Прокси)
    'recommended_hypotheses': hypotheses_v3,
}

os.makedirs(CLEANED_DIR, exist_ok=True)
pd.to_pickle(report_data, REPORT_DATA_PATH)

print(f'✓ Данные успешно сохранены в: {REPORT_DATA_PATH}')
print(f'Передано {len(report_data["recommended_hypotheses"])} ключевых гипотез (прокси-метрики) для презентации.')


✓ Данные успешно сохранены в: ../data/cleaned/report_data.pkl
Передано 3 ключевых гипотез (прокси-метрики) для презентации.


## Итоги и стратегические рекомендации

**Бизнес-модель: Высокая устойчивость и потенциал**
Анализ подтвердил, что текущая бизнес-модель обладает исключительным запасом прочности. Коэффициент **ROI Safety (LTV_safe / CAC) = 6.30** означает, что каждый вложенный в маркетинг евро приносит более 6 евро выручки уже на этапе первого платежа. Это фундамент для агрессивного масштабирования. Необходимо учитывать что здесь не рассматриваются оперативные и другие расходы. Мы можем говорить только о маркетинговых расходах. Но если в целом бизнес-модель устойчива, то правильные инвестиции в маркетинг необходимы и могут привести к значительному росту трафика клиентов.

**Ключевые выводы:**
1. **Эффективность привлечения (CAC/CPA):** При стоимости лида (CPA) в **8.08 €**, стоимость привлечения покупателя (CAC) составляет **183.69 €**. При среднем чеке (AOV) в **1,141 €**, бизнес остается глубоко прибыльным даже без учета последующих платежей и LTV.
2. **Конверсия (C1):** Текущий показатель **4.40%** является здоровым, но именно здесь скрыт основной рычаг роста прибыли. Рост C1 на 10% (до 4.8%) дает больший финансовый эффект, чем аналогичный рост объема трафика.
3. **Модель «Одной продажи»:** Показатель **APC = 1.01** указывает на то, что монетизация происходит в момент старта обучения. Это минимизирует риски кассовых разрывов, но ограничивает долгосрочный LTV текущими офферами.

**Рекомендации по развитию:**
1. **Фокус на Прокси-метрики:** Классические A/B тесты «в деньги» (до финальной оплаты) при текущем трафике занимают более 1.5 лет. Рекомендуется перейти к тестированию **прокси-метрик** (дозвон, запись на интенсив), где срок проверки гипотез сокращается до **7-14 дней**.
2. **Внедрение Лид-магнитов (Гипотеза H5):** Переход от прямой продажи курса к «входу через пользу» (бесплатные интенсивы/воркшопы) позволит кратно увеличить воронку на верхних этапах, сохраняя или снижая CAC.
3. **Автоматизация касаний (Гипотеза H4):** Внедрение мгновенного авто-ответа в мессенджеры (SLA < 2 мин) — самый быстрый способ поднять конверсию в контакт, что неизбежно приведет к росту итоговых продаж.

**Резюме:** Бизнес крепкий, прибыльный и готов к масштабированию. Основной вектор развития на ближайший квартал — системное повышение C1 через серию быстрых экспериментов с использованием прокси-метрик.